# 04 - TinyValue Exercises

这本 `course` 不是压缩版提纲。课堂 notebook 会先把直觉、例子、代码和图跑通，再进入最后的 Predict / Modify 检查。

学习顺序：先读这一页的主线，遇到代码就运行；最后再做底部的小检查。真正写作业时进入同目录的 `homework.ipynb`。

In [ ]:
from pathlib import Path
import sys, math, random, inspect


def find_repo_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / 'micrograd' / 'engine.py').exists():
            return q
    return p

ROOT = find_repo_root()
PROJECT_ROOT = ROOT
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from micrograd.engine import Value
from micrograd.nn import Neuron, Layer, MLP

try:
    import matplotlib.pyplot as plt
    MATPLOTLIB_AVAILABLE = True
except ModuleNotFoundError:
    plt = None
    MATPLOTLIB_AVAILABLE = False

try:
    import torch
except ModuleNotFoundError:
    torch = None


def near(a, b, eps=1e-6):
    return abs(a - b) < eps


def assert_close(actual, expected, name='value', eps=1e-9):
    assert abs(actual - expected) < eps, f'{name}: expected {expected}, got {actual}'


def todo_guard(values, message='请先填写 TODO，再运行检查。'):
    if any(v is None for v in values):
        print(message)
        return False
    return True

from course.checks import qa_check


def show_svg(path):
    path = Path(path)
    try:
        from IPython.display import SVG, display
        display(SVG(filename=str(path)))
    except Exception:
        print('图已生成，但当前环境不能内嵌显示。请打开:', path.resolve())

print('project root:', ROOT)


## 作业模式 vs 参考答案模式

```text
作业模式：先运行留空代码，看到 TODO 提示；然后自己补代码，再运行 qa_check。
提示模式：只展开 Show / Hide 提示，不看完整答案。
答案模式：最后才展开 Show / Hide 答案，对照自己的实现。
```

不要一口气照着完整答案写完。micrograd 的实现就是很多小局部规则拼起来的。

## 直觉地图：局部反传规则

| 运算 | 前向 | 局部导数 | `_backward` 里要做什么 |
|---|---|---|---|
| 加法 | `out = x + y` | `dout/dx=1`, `dout/dy=1` | 把 `out.grad` 原样加给两边 |
| 乘法 | `out = x * y` | `dout/dx=y`, `dout/dy=x` | 交叉乘以后再加给两边 |
| 幂 | `out = x**n` | `dout/dx=n*x**(n-1)` | 乘上 `out.grad` 加回 x |
| ReLU | `max(0,x)` | 正数是 1，负数是 0 | 正数传梯度，负数不传 |
| backward | 从 L 出发 | 链式法则 | topo 倒序调用每个 `_backward` |

## 公式速记

这一节写代码时只用到这几条局部导数：

$$
out = x + y \Rightarrow \frac{\partial out}{\partial x}=1, \frac{\partial out}{\partial y}=1
$$

$$
out = xy \Rightarrow \frac{\partial out}{\partial x}=y, \frac{\partial out}{\partial y}=x
$$

$$
out = x^n \Rightarrow \frac{\partial out}{\partial x}=n x^{n-1}
$$

$$
ReLU(x)=\max(0,x)
$$

```text
x > 0 时，ReLU 局部导数是 1
x < 0 时，ReLU 局部导数是 0
```

代码里的 `_backward` 就是在把这些局部导数乘上 `out.grad`。

## 从源码阅读过渡到自己写 TinyValue

04 的作业不是让你背 `Value` 源码，而是让你亲手实现一个最小自动求导对象。

一个运算节点要做两件事：

```text
forward：算出 out.data，并记住父节点和操作名
backward：等 out.grad 已经有值时，把梯度传回父节点
```

所以你写 `__add__`、`__mul__`、`relu`、`backward` 时，始终问自己两个问题：

```text
这个节点前向算出来的 data 是什么？
如果最终 loss 对 out 的梯度是 out.grad，我要给父节点加多少？
```

In [ ]:
class AddOnlyValue:
    def __init__(self, data, children=(), op=''):
        self.data = data
        self.grad = 0
        self._prev = set(children)
        self._op = op
        self._backward = lambda: None

    def __add__(self, other):
        other = other if isinstance(other, AddOnlyValue) else AddOnlyValue(other)
        out = AddOnlyValue(self.data + other.data, (self, other), '+')

        def _backward():
            self.grad += out.grad
            other.grad += out.grad

        out._backward = _backward
        return out

    def __repr__(self):
        return f'AddOnlyValue(data={self.data}, grad={self.grad})'


a = AddOnlyValue(2)
b = AddOnlyValue(3)
c = a + b
print('forward:', a, b, c)
print('c._op:', c._op)
print('c._prev data:', [v.data for v in c._prev])

c.grad = 1
c._backward()
print('after local backward:', a, b, c)

## 闭包到底记住了什么

上面 `_backward` 是在 `__add__` 里面定义的。它晚一点才执行，但它还记得：

```text
self  -> a
other -> b
out   -> c
```

这就是为什么 `_backward()` 被挂到 `out` 上以后，之后还能把 `out.grad` 分给 `a.grad` 和 `b.grad`。

你写 TinyValue 时，不要把闭包想神秘：它只是“内部函数记住了外层变量”。

In [ ]:
# 只手动演示一层反传，不做完整 topo。
x = AddOnlyValue(10)
y = AddOnlyValue(-4)
z = x + y
z.grad = 5
z._backward()
print('如果最终 loss 对 z 的梯度是 5：')
print('x.grad =', x.grad)
print('y.grad =', y.grad)
print('加法局部导数都是 1，所以两边都收到 5。')

## 完整 backward 为什么需要 topo

手动调用一层 `_backward()` 只适合 `c = a + b` 这种单层图。

一旦有：

```text
L = (a*b + a) * 2
```

你必须先让靠近 loss 的节点反传，再让更前面的节点反传。

这就是 topo 的作用：

```text
先递归收集父节点
最后 append 自己
反过来调用 _backward
```

所以 04 的完整作业，核心不是语法长，而是把“局部反传”按正确顺序串起来。

## What To Remember

这一节过关，只需要能讲清楚这 5 句话：

```text
1. 每个运算都要做前向：创建 out。
2. 每个运算都要给 out 绑定一个局部 _backward。
3. _backward 里用 +=，因为一个变量可能有多条路径。
4. backward() 要先 topo 排序，再倒序执行。
5. 一个完整 TinyValue 其实就是这些小规则拼起来。
```

## 课堂检查：先预测，再改一行

这一段保留 `course` 的隐藏检查。你应该先自己填，再展开提示或答案。

## Predict - 一个最小 Value 对象要存什么？

In [ ]:
# 填写说明：
# - 完成 student_minivalue_state，构造一个最小可求导节点的状态表。
# - 运行本 cell，看 qa_check 的提示。

def student_minivalue_state():
    # TODO: 返回 data/grad/_prev/_op/_backward 这五个最小状态。
    pass


qa_check('qa_check_class_04_predict', globals())


<details><summary>Show / Hide 本题提示</summary>

- `data` 是前向值，`grad` 初始是 0。
- `_prev` 保存父节点集合；`_backward` 一开始可以是什么都不做的函数。

</details>


<details><summary>Show / Hide 本题答案</summary>

```python
def student_minivalue_state():
    return {
        'data': 2.0,
        'grad': 0,
        '_prev': set(),
        '_op': '',
        '_backward': lambda: None,
    }
```

</details>


## Run - 一个只有 data/grad 的盒子

In [ ]:
class TinyBox:
    def __init__(self, data):
        self.data = data
        self.grad = 0

x = TinyBox(2.0)
print(x.data, x.grad)

## 拆开看 - 每个运算都要做两件事

```text
forward: 创建 out，算 out.data，记录父节点和 op
backward: 给父节点累加 grad
```

In [ ]:
print('add forward: out.data = self.data + other.data')
print('add backward: self.grad += out.grad; other.grad += out.grad')
print('mul forward: out.data = self.data * other.data')
print('mul backward: self.grad += other.data*out.grad; other.grad += self.data*out.grad')

## Modify - 只写加法反传

In [ ]:
# 填写说明：
# - 完成 student_add_backward_rule，只模拟加法节点如何把 out.grad 传给两个父节点。
# - 运行本 cell，看 qa_check 的提示。

def student_add_backward_rule(out_grad):
    # TODO: 加法对 self/other 的局部导数都是 1。
    self_contribution = None
    other_contribution = None
    return self_contribution, other_contribution


qa_check('qa_check_class_04_modify', globals())


<details><summary>Show / Hide 本题提示</summary>

- `out = self + other`。
- `dout/dself = 1`，`dout/dother = 1`，所以两边都拿到同一个 `out_grad`。

</details>


<details><summary>Show / Hide 本题答案</summary>

```python
def student_add_backward_rule(out_grad):
    self_contribution = out_grad
    other_contribution = out_grad
    return self_contribution, other_contribution
```

</details>
